# Agent 记忆系统 profiling：第一层（标准 benchmark），Colab L4

同一套 agent 负载（LoCoMo、LongMemEval-S 的用户消息按时间回放，最后问答）跑在 5 种记忆架构上：
RAG 对照组、mem0（嵌入式 Qdrant / pgvector / Qdrant server）、Graphiti（Neo4j）。推理用 llama.cpp（固定版本），
主模型 gpt-oss-20b，对照 Qwen3-14B、Qwen3-4B。

每个操作记录 LLM（prefill / decode / 排队 / 前缀缓存命中）、嵌入、数据库、块设备读写和 flush（fsync）、Postgres 落盘统计、GPU。
平台是云 GPU 虚拟机：存储结论按“云部署”解读（见 `docs/colab_plan.md`）。

**托管模式（默认）：** 运行时类型选 **L4 GPU** → 菜单“代码执行程序 → 全部运行”，之后不用再操作。
- 第 4 格是托管程序（`colab/supervisor.py`）：按 `colab/queue.txt` 依次跑实验，失败自动重试，卡住自动终止重来，
  每 10 分钟把进度、问题和已完成 run 的备份推送到仓库的 `colab-status` 分支（Claude 定期检查）。
- 要跑什么由 `colab/queue.txt` 决定（Claude 按你的决定修改），运行中的托管程序会自动拉取。
- 队列跑完、空闲 20 分钟后，确认 Drive 备份完整，然后自动释放运行时，不再消耗计算单元。
- 运行时断开（或被释放）后，再次“全部运行”即可：已完成的 run 从 Drive 恢复并跳过。
- **需要一次性设置：** Colab 左侧钥匙图标（Secrets）里加 `GH_TOKEN`，并打开“笔记本访问权限”。
  这个 token 在 GitHub → Settings → Developer settings → Fine-grained tokens 创建，只选这个仓库，权限 Contents：Read and write。

In [ ]:
#@title 1. 参数、GPU、Drive
import os, subprocess, time
PROFILE = "gptoss20b"            #@param ["gptoss20b", "q14b", "q4b"]
L4_PROFILES = "q14b q4b"         #@param {type:"string"}
CU_PER_HOUR = 0.0                #@param {type:"number"}
IDLE_MIN = 20                    #@param {type:"integer"}
AUTO_RELEASE = True              #@param {type:"boolean"}
# IDLE_MIN：队列跑完后等多少分钟（等新的队列行）再释放运行时；AUTO_RELEASE=False 则不释放
# CU_PER_HOUR：Colab 菜单“运行时 → 查看资源”里显示的每小时计算单元消耗，填上后花销表会换算成 CU
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu); assert gpu, "没有 GPU：运行时 → 更改运行时类型 → L4 GPU"
tag = gpu.split(",")[0].replace("NVIDIA ", "").replace(" ", "")
from google.colab import drive
drive.mount("/content/drive")
os.environ.update({
    "PROFILE": PROFILE, "L4_PROFILES": L4_PROFILES, "CU_PER_HOUR": str(CU_PER_HOUR),
    "OUT": f"/content/results/{tag}",                              # 本地盘（被测的盘）
    "SYNC_DIR": f"/content/drive/MyDrive/memprof/{tag}",          # 每组跑完镜像到 Drive，换运行时可续跑
    "MEMPROF_MODELS": "/root/models", "HOME": "/root",
    "IDLE_MIN": str(IDLE_MIN), "AUTO_RELEASE": "1" if AUTO_RELEASE else "0",
})
!nproc; free -g | head -2; df -h / | tail -1

In [ ]:
#@title 2. 代码（Colab Secrets 里的 GH_TOKEN：拉取代码 + 推送状态和备份）
BRANCH = "claude/gifted-maxwell-td8cnz"
try:
    from google.colab import userdata
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
if not tok:
    print("没有 GH_TOKEN：实验照常跑、结果照常存 Drive，但不会推送状态，Claude 无法远程检查")
url = f"https://{tok + '@' if tok else ''}github.com/envymoon/Agent_Mem_Sys.git"
if not os.path.exists("/content/Agent_Mem_Sys"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/Agent_Mem_Sys"], check=True)
else:
    subprocess.run(["git", "-C", "/content/Agent_Mem_Sys", "pull", "-q"], check=True)
%cd /content/Agent_Mem_Sys
!git log --oneline -1

In [ ]:
#@title 3. 安装：llama.cpp（CUDA）、Postgres/Qdrant/Neo4j、Python 环境、模型（约 23 GB）、数据集。约 25–35 分钟，可重复运行
t0 = time.time()
!DL_PROFILES="gptoss20b q14b q4b" bash scripts/local/setup.sh 2>&1 | grep -v -i warn | tail -25
os.environ["SESSION_SETUP_MIN"] = f"{(time.time() - t0) / 60:.0f}"
print(f"setup {(time.time() - t0) / 60:.1f} min")

In [ ]:
#@title 4. 托管运行（这一格会一直运行到队列跑完；中途停止这一格不会停止实验，重新运行即可接管）
%run colab/supervisor.py


In [ ]:
#@title 4b. （可选）手动查看进度：先停止第 4 格（实验在后台继续），看完再重新运行第 4 格
!tail -n 30 $OUT/suite.log; nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total,power.draw --format=csv


## 5. 看效果和花销，再决定跑什么

- `qa_summary.md`：各系统在 LoCoMo / LongMemEval 上的 F1 和 LLM 评判准确率（评判模型是本地模型，只适合系统间比较）。
- `budget.md`：根据 pilot 外推全套实验各需要多少小时（填了 `CU_PER_HOUR` 就同时给出 CU）。
- `breakdown.md`：每个操作的耗时分解（LLM / 嵌入 / 数据库 / flush / 读写字节）。

In [ ]:
#@title 5. 效果、花销、耗时分解
runs = !ls -d $OUT/l1_*/ $OUT/lme_*/ 2>/dev/null
!.venv/bin/python analysis/multi_breakdown.py {" ".join(runs)} > $OUT/breakdown.md 2>/dev/null; cp $OUT/breakdown.md $SYNC_DIR/ 2>/dev/null
!cat $OUT/qa_summary.md; echo; cat $OUT/budget.md; echo; cat $OUT/breakdown.md | head -80

## 6. 全套实验（托管模式下不用这一格：在 `colab/queue.txt` 里加一行即可。下面是手动方式）

| 套件 | 内容 |
|---|---|
| `l1` | 5 个系统 × LoCoMo 100 轮 + 50 题，同步写入（主结果） |
| `lme` | 3 个 LongMemEval-S 完整历史（跨会话 / 时间推理 / 知识更新各一题）× 5 个系统 |
| `l2` | 后台异步写对前台的干扰（思考时间 0 s / 5 s） |
| `l3` | mem0-pg、Graphiti × 6 种 KV 配置（关前缀缓存、关内存缓存、KV q8、KV 放内存、8k 上下文） |
| `l4` | 换模型：Qwen3-14B、Qwen3-4B |
| `l5` | 存储规模 10³–10⁵ 条，冷 / 热页缓存 |
| `model` | 延迟模型：本机预测误差、换 GPU / 换盘推算 |
| `score` | 对所有 run 打分 |

In [ ]:
#@title 6. 选要跑的套件（空格分隔，按顺序执行）
SUITES = "l1 lme score"   #@param {type:"string"}
RUN_THIS = False          #@param {type:"boolean"}
# 托管模式下保持 False（“全部运行”时不会误启动）；要手动跑时勾选
if RUN_THIS:
    !nohup scripts/local/run_local_suite.sh {SUITES} > /content/suite_console.log 2>&1 &
    print("started:", SUITES)

In [ ]:
#@title 7. （可选）在另一个运行时重复关键实验，估计不同虚拟机之间的波动
REP = 2   #@param {type:"integer"}
RUN_THIS = False   #@param {type:"boolean"}
if RUN_THIS:
    !REP={REP} TURNS=100 QA=50 SYSTEMS="mem0-pg graphiti rag-pg" nohup scripts/local/run_local_suite.sh preflight l1 > /content/rep_console.log 2>&1 &